# PHYCOM V3.2 — final validation code for Kaggle T4 ×2

**Import this notebook, select GPU T4 ×2, enable Internet, then choose Run All.**
The complete readable Python project is embedded below. No separate code or dataset upload is required.

This revision adds the controls needed after reviewing your V3.1 results:

- Up to **240 total epochs**, with the same validation stopping rule, and a separate **60-epoch reference**.
- A matched **64-unit output-head comparison** for gated and ungated 4×4 CNN–LSTM.
- Gated/ungated robustness tests, a classical comparator using the same reported SNR, and **three seeds per sequence window**.
- **Fresh confirmation channels**, generated only after all validation-selected models are frozen.
- Computed paper tables, PDF/PNG figures, raw block statistics, paired uncertainty and convergence diagnostics.

The full study contains **129 fits**. It automatically reuses your audited V3.1 checkpoints if you attach the compatible old results ZIP as an input. Otherwise it trains from scratch. Your old files remain unchanged.

The target is the setup that succeeded in your last run: Python 3.12, PyTorch 2.10.0+cu128, CUDA 12.8 and two T4 GPUs. Official Mamba and causal-conv1d binary-wheel setup is retained, with CUDA training/resume checks before the experiment.

**Validation scope:** local mathematical/checkpoint/installer tests and a complete CPU smoke workflow were run during development. Full Mamba/CUDA training must run on your Kaggle GPUs. Completion, convergence, improved accuracy and publication acceptance are separate questions; the final report distinguishes them.

Allow several hours. The default session budget is 10.5 hours; if needed, the notebook creates a resumable partial archive. It cannot guarantee finishing before Kaggle's session limit.


## 1. Settings

Leave these defaults for the full study. Optionally attach `PHYCOM_V3_1_PUBLICATION_RESULTS (1).zip` as a Kaggle input; it is detected automatically. A path to an extracted V3.1 result directory also works.

For a later **V3.2** continuation, set `RESUME_ARCHIVE` to your downloaded partial/results ZIP. Start a fresh GPU session for the full budget. `SMOKE_TEST=True` runs a small non-Mamba diagnostic whose numbers must not be used in the paper.


In [ ]:
from pathlib import Path
import os

SMOKE_TEST = False
USE_BOTH_GPUS = True
PARENT_V31 = "auto"       # Optional audited V3.1 result ZIP/directory, "auto", or "none"
RESUME_ARCHIVE = ""       # Optional path to your own V3.2 PARTIAL/RESULTS ZIP
SESSION_HOURS = 10.5       # Graceful pause budget for THIS invocation; 0 disables

WORKING = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
WORKING = WORKING.resolve()
PROJECT = WORKING/'PHYCOM_V3_2_CODE'
OUTPUT = WORKING/('PHYCOM_V3_2_SMOKE' if SMOKE_TEST else 'PHYCOM_V3_2_PUBLICATION')
os.chdir(WORKING)
print('Mode:', 'SMOKE DIAGNOSTIC' if SMOKE_TEST else 'FULL CONFIRMATION STUDY')
print('Output:', OUTPUT)


## 2. Extract the complete source project

This cell verifies the embedded archive before extraction. The extracted Python files, tests and documentation are the same project as the companion GitHub ZIP. Compatible existing source is reused. Existing files with different content are preserved and cause a clear stop.


In [ ]:
import base64, hashlib, io, zipfile

PAYLOAD_SHA256 = "edf774b378753ac727f466a2b17087b7e8e1905ee5656d20e8f87f4dc18f930f"
PAYLOAD_BASE64 = """
...
ABkAGQAbBgAAlwMBAAAA
"""
blob = base64.b64decode(PAYLOAD_BASE64)
if hashlib.sha256(blob).hexdigest() != PAYLOAD_SHA256:
    raise RuntimeError('Embedded source checksum failed. Re-import the original notebook.')
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    for member in archive.infolist():
        if member.is_dir():
            continue
        target = (PROJECT/member.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()):
            raise ValueError('Unsafe path inside source archive')
        content = archive.read(member)
        if target.exists() and target.read_bytes() != content:
            raise RuntimeError(f'Existing source differs: {target}. '
                               'Keep it and choose a new PROJECT directory, or start a fresh Kaggle session.')
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)
print('Verified and extracted:', PROJECT)
print('Source archive SHA-256:', PAYLOAD_SHA256)


## 3. Set up and Run All

Setup uses checked official binary wheels; it does not compile CUDA extensions or replace the existing core numerical stack. Downloading and CPU data preparation can show zero GPU utilization. Once training starts, the scheduler uses one independent job per GPU and prints progress.

All models are trained first. A frozen checkpoint manifest is then written, followed by fresh evaluation, isolated timing, figures and an arithmetic audit. Checkpoints are committed after every epoch; evaluated cases are saved individually.

If the session budget is reached, download the `_PARTIAL.zip`, attach it to a fresh notebook, set `RESUME_ARCHIVE`, and run the same notebook again. A stopped or partial run is not presented as a completed publication experiment.


In [ ]:
import importlib.util
import signal
import subprocess
import sys
from IPython.display import display, FileLink

command = [sys.executable, '-u', str(PROJECT/'run_phycom.py'),
           '--output', str(OUTPUT), '--parent', str(PARENT_V31),
           '--session-hours', str(SESSION_HOURS)]
if SMOKE_TEST:
    command += ['--smoke', '--no-mamba']
if not USE_BOTH_GPUS:
    command += ['--single-gpu']
if RESUME_ARCHIVE:
    command += ['--resume', str(RESUME_ARCHIVE)]

RUN_STATUS = 'running'
print('Starting setup and experiment. Progress appears below.', flush=True)
process = subprocess.Popen(command, cwd=PROJECT, start_new_session=True)
try:
    returncode = process.wait()
except KeyboardInterrupt:
    print('Interrupt received. Asking the experiment to stop and archive committed work.', flush=True)
    process.send_signal(signal.SIGINT)
    try:
        returncode = process.wait(timeout=120)
    except subprocess.TimeoutExpired:
        process.terminate()
        returncode = process.wait(timeout=30)
    RUN_STATUS = 'interrupted'
else:
    RUN_STATUS = 'complete' if returncode == 0 else 'paused' if returncode == 75 else 'failed'

if RUN_STATUS in ('paused', 'interrupted'):
    print('Saved work can resume. The next cell shows available archives.')
elif RUN_STATUS == 'failed':
    spec = importlib.util.spec_from_file_location('phycom_diagnostics', PROJECT/'diagnostics.py')
    diagnostic_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(diagnostic_module)
    diagnostic = diagnostic_module.create_diagnostics(PROJECT, WORKING, OUTPUT)
    display(FileLink(diagnostic.name))
    partial = OUTPUT.parent/(OUTPUT.name+'_PARTIAL.zip')
    if partial.is_file():
        display(FileLink(partial.name))
    raise RuntimeError(f'Experiment stopped with exit code {returncode}. '
                       'Read the original error above; diagnostics and committed work are downloadable.')


## 4. Download and use the computed evidence

A successful full run saves `PHYCOM_V3_2_PUBLICATION_RESULTS.zip`. Its `Paper_Data/RESULTS_SUMMARY.md` explains what the new results support. Use its computed CSV/LaTeX tables and `Figures/` for the manuscript. Read `convergence.csv` before describing a model as converged.

`FINAL_AUDIT.json` verifies completion, frozen checkpoint identity and reproduction of scalar metrics from raw statistics. It is not a journal-acceptance certificate. Keep any stronger classical baseline and seed reversals in your conclusions.


In [ ]:
import json
from IPython.display import display, FileLink, Markdown

manifest_path = OUTPUT/'run_manifest.json'
audit_path = OUTPUT/'FINAL_AUDIT.json'
manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
audit = json.loads(audit_path.read_text()) if audit_path.is_file() else {}
results = OUTPUT.parent/(OUTPUT.name+'_RESULTS.zip')
partial = OUTPUT.parent/(OUTPUT.name+'_PARTIAL.zip')
diagnostic = WORKING/'PHYCOM_V3_2_DIAGNOSTICS.zip'

if manifest.get('status') == 'complete' and audit.get('checks_passed') and results.is_file():
    print('COMPLETE:', audit['total_fits'], 'fits;', audit['metric_rows'], 'metric rows.')
    print('Fits requiring convergence review:', audit.get('convergence_review_fits', 'see report'))
    display(FileLink(results.name))
    guide = OUTPUT/'Paper_Data'/'RESULTS_SUMMARY.md'
    display(FileLink(str(guide.relative_to(WORKING))))
    if not audit.get('publication_run'):
        print('SMOKE TEST ONLY: these small-run results must not be used in the manuscript.')
    else:
        print('Read the results guide and convergence report before finalizing manuscript claims.')
else:
    print('Run status:', manifest.get('status', RUN_STATUS if 'RUN_STATUS' in globals() else 'not started'))
    if partial.is_file():
        display(FileLink(partial.name))
        print('To resume: attach this partial ZIP to the same notebook in a fresh session and set RESUME_ARCHIVE.')
    if diagnostic.is_file():
        display(FileLink(diagnostic.name))
    if not partial.is_file() and not diagnostic.is_file():
        print('No archive is available yet. Check the original setup error above.')
print('Readable GitHub source:', PROJECT)
print('Methods documentation:', PROJECT/'METHODS.md')
